# Project 3: Galaxy Spatial Distribution & Overdensities

**Difficulty:** Advanced  
**Time:** 30-35 minutes  
**ICARD-MU Workshop | Day 2**

---

## Learning Objectives
- Visualize the **large-scale structure** of the Universe
- Learn spatial analysis techniques:
  - Surface density calculation
  - Nearest-neighbor methods
  - Overdensity identification
- Identify **galaxy groups and proto-clusters**
- Investigate **environmental effects** on galaxy evolution
- Connect to real research on cosmic web!

---

## Background

### The Cosmic Web

Galaxies are not randomly distributed - they trace the **cosmic web**:
- **Clusters**: Dense regions with hundreds-thousands of galaxies
- **Groups**: Smaller overdensities (10-50 galaxies)
- **Filaments**: Structures connecting clusters
- **Voids**: Underdense regions

### Why Study Environment?

**Environment affects galaxy evolution!**
- Dense regions → more quiescent galaxies
- Galaxy-galaxy interactions
- Ram-pressure stripping
- Strangulation/starvation

This is directly related to your research on COSMOS overdensities!

## Step 1: Import Libraries and Load Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy import units as u
from scipy.spatial import cKDTree
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set nice plotting defaults
plt.rcParams['figure.figsize'] = (12, 10)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1.5

print("Libraries loaded successfully!")

In [ ]:
# Load COSMOS-Web catalog
print("Loading COSMOS-Web master catalog...")
print("This may take a minute...\n")

catalog_path = '/Volumes/Ronaldo_SSD/COSMOSWEB/COSMOSWeb_mastercatalog_v1.fits'

with fits.open(catalog_path) as hdu:
    # Create master catalog
    master_catalog = Table()
    
    # HDU 1: Basic info and coordinates
    print("Loading coordinates...")
    master_catalog['id'] = hdu[1].data['id']
    master_catalog['ra'] = hdu[1].data['ra']   # degrees
    master_catalog['dec'] = hdu[1].data['dec']  # degrees
    
    # HDU 2: LePhare - redshifts and physical parameters
    print("Loading physical parameters...")
    master_catalog['z_best'] = hdu[2].data['zfinal']
    master_catalog['z_med'] = hdu[2].data['zpdf_med']
    master_catalog['mass'] = hdu[2].data['mass_med']  # log10(M☉)
    master_catalog['sfr'] = hdu[2].data['sfr_med']    # log10(M☉/yr)
    master_catalog['ssfr'] = hdu[2].data['ssfr_med']  # log10(yr^-1)

print(f"\n✓ Loaded {len(master_catalog)} galaxies")
print(f"✓ RA range: {np.min(master_catalog['ra']):.4f}° - {np.max(master_catalog['ra']):.4f}°")
print(f"✓ Dec range: {np.min(master_catalog['dec']):.4f}° - {np.max(master_catalog['dec']):.4f}°")

## Step 2: Quality Control and Redshift Slicing

For spatial analysis, we'll focus on a narrow redshift slice to study structure at a specific cosmic epoch.

In [ ]:
# Quality cuts
print("Applying quality cuts...\n")

# 1. Valid measurements
valid_mask = (
    (master_catalog['mass'] > 9.0) & 
    (master_catalog['mass'] < 20) &
    np.isfinite(master_catalog['ra']) &
    np.isfinite(master_catalog['dec']) &
    np.isfinite(master_catalog['z_best'])
)

# 2. Select a redshift slice (z = 1.0 - 1.5)
# This is a good range with many galaxies
z_min, z_max = 1.0, 1.5
z_mask = (master_catalog['z_best'] >= z_min) & (master_catalog['z_best'] <= z_max)

# Combine masks
final_mask = valid_mask & z_mask
clean_sample = master_catalog[final_mask]

print(f"Redshift slice: {z_min} < z < {z_max}")
print(f"✓ Sample size: {len(clean_sample)} galaxies")
print(f"✓ Median redshift: {np.median(clean_sample['z_best']):.3f}")
print(f"✓ Median mass: 10^{np.median(clean_sample['mass']):.2f} M☉")

## Step 3: Visualize the Sky Distribution

Let's see where galaxies are located on the sky!

In [ ]:
# Extract coordinates
ra = clean_sample['ra']
dec = clean_sample['dec']

# Create sky map
fig, ax = plt.subplots(figsize=(14, 10))

# 2D histogram (density map)
hist, xedges, yedges = np.histogram2d(ra, dec, bins=50)
extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]

im = ax.imshow(hist.T, 
               origin='lower', 
               extent=extent, 
               cmap='hot', 
               aspect='auto',
               interpolation='gaussian')

cbar = plt.colorbar(im, ax=ax, label='Number of Galaxies per Bin')

ax.set_xlabel('Right Ascension (degrees)', fontsize=16)
ax.set_ylabel('Declination (degrees)', fontsize=16)
ax.set_title(f'COSMOS-Web Sky Distribution ({z_min} < z < {z_max})', 
             fontsize=18, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--', color='white')

plt.tight_layout()
plt.show()

print("\nQuestion: Do you see any obvious overdense regions?")
print("These could be galaxy groups or proto-clusters!")

## Step 4: Calculate Surface Density

We'll use the **Nth nearest neighbor** method to calculate local surface density for each galaxy.

In [ ]:
# Build KD-tree for fast nearest neighbor search
print("Building spatial index (KD-tree)...")

# Convert RA, Dec to Cartesian coordinates (approximate for small fields)
# For better accuracy, use proper angular distance, but this works for COSMOS
coords = np.column_stack([ra, dec])
tree = cKDTree(coords)

# Find 5th nearest neighbor distance for each galaxy
N_neighbors = 5  # Including the galaxy itself
print(f"Finding {N_neighbors}th nearest neighbor for each galaxy...")

distances, indices = tree.query(coords, k=N_neighbors+1)  # +1 to exclude self
# distances[:, 0] is the galaxy itself (distance = 0)
# distances[:, N_neighbors] is the Nth nearest neighbor

nth_distance = distances[:, N_neighbors]  # in degrees

# Convert to arcmin
nth_distance_arcmin = nth_distance * 60.0

# Calculate surface density
# Σ = N / (π * r^2)
# Units: galaxies per arcmin^2
surface_density = N_neighbors / (np.pi * nth_distance_arcmin**2)

print(f"✓ Calculated surface density for {len(clean_sample)} galaxies")
print(f"\nSurface Density Statistics:")
print(f"  Median: {np.median(surface_density):.4f} gal/arcmin²")
print(f"  Mean: {np.mean(surface_density):.4f} gal/arcmin²")
print(f"  Min: {np.min(surface_density):.4f} gal/arcmin²")
print(f"  Max: {np.max(surface_density):.4f} gal/arcmin²")

## Step 5: Map Galaxy Overdensities

Let's define **overdensity** as δ = (Σ - Σ_median) / Σ_median

In [ ]:
# Calculate overdensity
median_density = np.median(surface_density)
overdensity = (surface_density - median_density) / median_density

# Add to catalog
clean_sample['surface_density'] = surface_density
clean_sample['overdensity'] = overdensity

print("Overdensity Statistics:")
print(f"  Median δ: {np.median(overdensity):.3f}")
print(f"  Mean δ: {np.mean(overdensity):.3f}")
print(f"  Std δ: {np.std(overdensity):.3f}")
print(f"  Range: {np.min(overdensity):.2f} to {np.max(overdensity):.2f}")

In [ ]:
# Create overdensity map
fig, ax = plt.subplots(figsize=(14, 10))

# Scatter plot colored by overdensity
scatter = ax.scatter(ra, dec, 
                     c=overdensity, 
                     cmap='RdBu_r', 
                     s=20, 
                     alpha=0.7,
                     vmin=-1, 
                     vmax=3,
                     edgecolors='none',
                     rasterized=True)

cbar = plt.colorbar(scatter, ax=ax, label='Overdensity δ = (Σ - Σ_med) / Σ_med')
cbar.ax.axhline(y=0, color='black', linewidth=2, linestyle='--')
cbar.ax.text(0.5, 0.05, 'Median', fontsize=10, va='center')

ax.set_xlabel('Right Ascension (degrees)', fontsize=16)
ax.set_ylabel('Declination (degrees)', fontsize=16)
ax.set_title(f'Galaxy Overdensity Map ({z_min} < z < {z_max})', 
             fontsize=18, fontweight='bold')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3, linestyle='--')

# Add text annotations
ax.text(0.02, 0.98, f'N = {len(clean_sample)} galaxies', 
        transform=ax.transAxes, fontsize=12, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

print("\nRed regions = overdense (groups/clusters)")
print("Blue regions = underdense (field/voids)")

## Step 6: Identify High-Density Structures

Let's find the most significant overdensities (δ > 2σ)

In [ ]:
# Define overdensity threshold (2σ above median)
overdensity_threshold = 2.0 * np.std(overdensity)

high_density_mask = overdensity > overdensity_threshold
low_density_mask = overdensity < -1.0
field_mask = ~high_density_mask & ~low_density_mask

n_high = np.sum(high_density_mask)
n_low = np.sum(low_density_mask)
n_field = np.sum(field_mask)

print(f"Environmental Classification (δ threshold = {overdensity_threshold:.2f}):")
print(f"  High-density regions:  {n_high:5d} ({100*n_high/len(clean_sample):.1f}%)")
print(f"  Field:                 {n_field:5d} ({100*n_field/len(clean_sample):.1f}%)")
print(f"  Low-density regions:   {n_low:5d} ({100*n_low/len(clean_sample):.1f}%)")
print(f"  Total:                 {len(clean_sample):5d}")

In [ ]:
# Plot with environmental classification
fig, ax = plt.subplots(figsize=(14, 10))

# Plot different environments
ax.scatter(ra[low_density_mask], dec[low_density_mask], 
           c='blue', s=15, alpha=0.3, label=f'Low-density ({n_low})', rasterized=True)
ax.scatter(ra[field_mask], dec[field_mask], 
           c='gray', s=15, alpha=0.2, label=f'Field ({n_field})', rasterized=True)
ax.scatter(ra[high_density_mask], dec[high_density_mask], 
           c='red', s=30, alpha=0.7, edgecolors='darkred', linewidth=0.5,
           label=f'High-density ({n_high})', rasterized=True)

ax.set_xlabel('Right Ascension (degrees)', fontsize=16)
ax.set_ylabel('Declination (degrees)', fontsize=16)
ax.set_title(f'Galaxy Groups and Overdensities ({z_min} < z < {z_max})', 
             fontsize=18, fontweight='bold')
ax.set_aspect('equal')
ax.legend(loc='upper right', fontsize=12, framealpha=0.9, markerscale=2)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

## Step 7: Environmental Effects on Galaxy Properties

**Key Science Question:** Do galaxies in dense regions have different properties than field galaxies?

In [ ]:
# Extract physical properties
log_mass = clean_sample['mass']
log_sfr = clean_sample['sfr']
log_ssfr = clean_sample['ssfr']

# Compare properties in different environments
print("Galaxy Properties vs Environment:\n")
print("                     High-Density       Field         Low-Density")
print("                     ------------    -----------    -------------")

median_mass_high = np.median(log_mass[high_density_mask])
median_mass_field = np.median(log_mass[field_mask])
median_mass_low = np.median(log_mass[low_density_mask])
print(f"Median log(M*):      {median_mass_high:6.2f}         {median_mass_field:6.2f}        {median_mass_low:6.2f}")

median_sfr_high = np.median(log_sfr[high_density_mask])
median_sfr_field = np.median(log_sfr[field_mask])
median_sfr_low = np.median(log_sfr[low_density_mask])
print(f"Median log(SFR):     {median_sfr_high:6.2f}         {median_sfr_field:6.2f}        {median_sfr_low:6.2f}")

median_ssfr_high = np.median(log_ssfr[high_density_mask])
median_ssfr_field = np.median(log_ssfr[field_mask])
median_ssfr_low = np.median(log_ssfr[low_density_mask])
print(f"Median log(sSFR):   {median_ssfr_high:6.2f}        {median_ssfr_field:6.2f}       {median_ssfr_low:6.2f}")

# Calculate quiescent fractions
quiescent_cutoff = -11  # log(sSFR) < -11
fq_high = np.sum(log_ssfr[high_density_mask] < quiescent_cutoff) / n_high
fq_field = np.sum(log_ssfr[field_mask] < quiescent_cutoff) / n_field
fq_low = np.sum(log_ssfr[low_density_mask] < quiescent_cutoff) / n_low
print(f"Quiescent frac:      {fq_high:6.2%}         {fq_field:6.2%}        {fq_low:6.2%}")

In [ ]:
# Plot sSFR distributions for different environments
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: sSFR histograms
ax = axes[0]
bins = np.linspace(-13, -8, 30)

ax.hist(log_ssfr[high_density_mask], bins=bins, 
        alpha=0.7, color='red', label=f'High-density ({n_high})', 
        density=True, histtype='stepfilled', linewidth=2)
ax.hist(log_ssfr[field_mask], bins=bins, 
        alpha=0.5, color='gray', label=f'Field ({n_field})', 
        density=True, histtype='step', linewidth=2)
ax.hist(log_ssfr[low_density_mask], bins=bins, 
        alpha=0.7, color='blue', label=f'Low-density ({n_low})', 
        density=True, histtype='step', linewidth=2, linestyle='--')

ax.axvline(quiescent_cutoff, color='black', linestyle='--', linewidth=2, alpha=0.7)
ax.text(quiescent_cutoff-0.2, ax.get_ylim()[1]*0.9, 'Quiescent', 
        fontsize=11, rotation=90, va='top')

ax.set_xlabel(r'$\log_{10}(\mathrm{sSFR} / \mathrm{yr}^{-1})$', fontsize=14)
ax.set_ylabel('Normalized Frequency', fontsize=14)
ax.set_title('sSFR Distribution vs Environment', fontsize=16, fontweight='bold')
ax.legend(fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')

# Panel 2: Quiescent fraction vs overdensity
ax = axes[1]

# Bin by overdensity
delta_bins = np.linspace(-1.5, 4, 10)
delta_centers = []
quiescent_fracs = []
quiescent_errs = []

for i in range(len(delta_bins)-1):
    mask = (overdensity >= delta_bins[i]) & (overdensity < delta_bins[i+1])
    n_bin = np.sum(mask)
    
    if n_bin > 5:
        n_q = np.sum(mask & (log_ssfr < quiescent_cutoff))
        frac = n_q / n_bin
        err = np.sqrt(frac * (1-frac) / n_bin)
        
        delta_centers.append(0.5 * (delta_bins[i] + delta_bins[i+1]))
        quiescent_fracs.append(frac)
        quiescent_errs.append(err)

ax.errorbar(delta_centers, quiescent_fracs, yerr=quiescent_errs,
            fmt='o-', markersize=10, linewidth=2.5, capsize=5, capthick=2,
            color='darkred', label='COSMOS-Web')

ax.set_xlabel('Overdensity δ', fontsize=14)
ax.set_ylabel('Quiescent Fraction', fontsize=14)
ax.set_title('Environmental Quenching', fontsize=16, fontweight='bold')
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1.5, alpha=0.5)
ax.legend(fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print("\nKey Result: Do you see environmental quenching?")
print("Higher quiescent fraction in dense regions suggests environment matters!")

## Step 8: SFMS in Different Environments

In [ ]:
# Plot SFMS for different environments
fig, ax = plt.subplots(figsize=(11, 8))

# Filter out non-star-forming galaxies for clarity
sf_mask = log_sfr > -2.0

# Plot different environments
ax.scatter(log_mass[low_density_mask & sf_mask], log_sfr[low_density_mask & sf_mask], 
           c='blue', s=20, alpha=0.4, label='Low-density', rasterized=True)
ax.scatter(log_mass[field_mask & sf_mask], log_sfr[field_mask & sf_mask], 
           c='gray', s=15, alpha=0.2, label='Field', rasterized=True)
ax.scatter(log_mass[high_density_mask & sf_mask], log_sfr[high_density_mask & sf_mask], 
           c='red', s=25, alpha=0.5, edgecolors='darkred', linewidth=0.3,
           label='High-density', rasterized=True)

# Fit SFMS for each environment
from scipy import stats

# High-density SFMS
if np.sum(high_density_mask & sf_mask) > 10:
    slope_h, intercept_h, r_h, p_h, se_h = stats.linregress(
        log_mass[high_density_mask & sf_mask], 
        log_sfr[high_density_mask & sf_mask]
    )
    mass_fit = np.linspace(9, 12, 100)
    ax.plot(mass_fit, slope_h * mass_fit + intercept_h, 
            'r-', linewidth=3, alpha=0.8, label='High-density MS')

# Field SFMS
if np.sum(field_mask & sf_mask) > 10:
    slope_f, intercept_f, r_f, p_f, se_f = stats.linregress(
        log_mass[field_mask & sf_mask], 
        log_sfr[field_mask & sf_mask]
    )
    ax.plot(mass_fit, slope_f * mass_fit + intercept_f, 
            'gray', linewidth=3, alpha=0.8, linestyle='--', label='Field MS')

ax.set_xlabel(r'$\log_{10}(M_* / M_{\odot})$', fontsize=16)
ax.set_ylabel(r'$\log_{10}(\mathrm{SFR} / M_{\odot}\,\mathrm{yr}^{-1})$', fontsize=16)
ax.set_title('Star-Forming Main Sequence vs Environment', fontsize=18, fontweight='bold')
ax.set_xlim(9, 12)
ax.set_ylim(-2, 3)
ax.legend(loc='lower right', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print("\nQuestion: Is the SFMS different in dense vs field environments?")
print("Do you see more scatter or offset in high-density regions?")

## Step 9: Summary Statistics

In [ ]:
print("="*70)
print("PROJECT 3 SUMMARY: Galaxy Spatial Distribution & Overdensities")
print("="*70)
print(f"\nSample: {len(clean_sample)} galaxies ({z_min} < z < {z_max})")
print(f"\nEnvironmental Classification:")
print(f"  High-density (δ > {overdensity_threshold:.2f}):  {n_high:5d} ({100*n_high/len(clean_sample):.1f}%)")
print(f"  Field:                                {n_field:5d} ({100*n_field/len(clean_sample):.1f}%)")
print(f"  Low-density (δ < -1.0):               {n_low:5d} ({100*n_low/len(clean_sample):.1f}%)")
print(f"\nEnvironmental Effects:")
print(f"  Quiescent fraction (high-density): {fq_high:.1%}")
print(f"  Quiescent fraction (field):        {fq_field:.1%}")
print(f"  Quiescent fraction (low-density):  {fq_low:.1%}")
print(f"\n  Δ(Quiescent fraction) = {(fq_high - fq_field)*100:.1f}% (high vs field)")
print(f"\nMedian sSFR:")
print(f"  High-density: {median_ssfr_high:.2f}")
print(f"  Field:        {median_ssfr_field:.2f}")
print(f"  Low-density:  {median_ssfr_low:.2f}")
print("\n" + "="*70)
print("\nKey Takeaway: Environment plays a significant role in galaxy evolution!")
print("Dense regions show enhanced quenching - this is environmental quenching.")

## Discussion Questions

1. **What causes environmental quenching?**
   - Ram-pressure stripping (hot gas removed)
   - Strangulation (gas supply cut off)
   - Galaxy harassment (tidal interactions)
   - AGN feedback

2. **How do we distinguish these mechanisms?**
   - Morphology changes
   - Gas content
   - Star formation history
   - Spatial distribution

3. **What is the role of mass vs environment?**
   - Mass quenching vs environmental quenching
   - Are they independent or coupled?
   - Pre-processing in groups before cluster infall

4. **How does this relate to your research?**
   - COSMOS overdensities at high-z
   - Proto-cluster evolution
   - When did environmental quenching become important?

---

## Next Steps & Research Ideas

**Extending this analysis:**
1. Compare different redshift slices - evolution of environmental effects
2. Add morphology - do dense regions have more spheroids?
3. Measure radial profiles around massive galaxies
4. Cross-match with X-ray data - find real clusters
5. Study velocity dispersions (with spectroscopy)

**Connection to research:**
- This technique is used to find proto-clusters at high-z
- Environmental studies are crucial for understanding galaxy evolution
- Your work on COSMOS overdensities uses similar methods!

---

## Congratulations!

You've completed all 3 projects and learned:
1. **SFMS** - Fundamental galaxy scaling relations
2. **UVJ** - Color-based galaxy classification
3. **Environment** - Large-scale structure and environmental effects

These are the building blocks of modern galaxy evolution research!

**Keep exploring JWST data - the Universe awaits!** 🌌